In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd

# Define constants
plate_type = "AssayPlate_Corning_3830"
cell_line = "HT29"

# Define wells
rows = [chr(i) for i in range(ord('B'), ord('O') + 1)]
cols = list(range(3, 23))

# Concentration series
concentrations = [10, 3.1642857, 1, 0.31428571, 0.1,
                  0.0316473, 0.010002279, 0.003167009,
                  0.001002506, 0.000316388]

# Image IDs, cp_ids, and plate names
plates = {
    "5514": {"cp_ids": ["8905", "9004", "9005", "9006", "9010"], "barcode": "CellPainting_20241220clearedspheroidsBOMI_20241220_151510"},
    "6072": {"cp_ids": ["9591", "9592", "9593", "9594", "9607"], "barcode": "CellPainting_20250127Cellpaintcleared3D_20250127_171120"},
    "6073": {"cp_ids": ["9596", "9597", "9598", "9599"], "barcode": "CellPainting_CellPaint3DBomi_WI_for_Jordi_20250203_155142"}
}

records = []
for image_id, info in plates.items():
    for cp_id in info["cp_ids"]:
        for row in rows:
            for col in cols:
                well_id = f"{row}{col:02d}"
                rec = {
                    "barcode": info["barcode"],
                    "image_id": image_id,
                    "cp_id": cp_id,
                    "well_id": well_id,
                    "plate_type": plate_type,
                    "cell_line": cell_line,
                    "cmpdname": None,
                    "solvent": None,
                    "cmpd_conc": None,
                    "cmpd_conc_unit": None,
                    "pert_type": None
                }
                
                if row in ['B', 'O']:
                    rec.update({
                        "cmpdname": "water",
                        "solvent": "water",
                        "cmpd_conc": 0,
                        "cmpd_conc_unit": "%",
                        "pert_type": "neg_con"
                    })
                elif col <= 12:
                    idx = col - 3; conc = concentrations[idx]
                    rec.update({"cmpd_conc": conc, "cmpd_conc_unit": "mM"})
                    if row in ['C', 'D', 'E']:
                        rec.update({"cmpdname": "nocodazole", "solvent": "DMSO", "pert_type": "pos_con"})
                    elif row in ['F', 'G', 'H']:
                        rec.update({"cmpdname": "berberine chloride", "solvent": "DMSO", "pert_type": "pos_con"})
                    elif row in ['I', 'J', 'K']:
                        rec.update({"cmpdname": "sorbitol", "solvent": "DMSO", "pert_type": "pos_con"})
                    elif row in ['L', 'M', 'N']:
                        rec.update({"cmpdname": "fluphenazine", "solvent": "DMSO", "pert_type": "pos_con"})
                else:
                    idx = col - 13
                    if row in ['C', 'D', 'E']:
                        conc = concentrations[idx]
                        rec.update({"cmpdname": "etoposide", "solvent": "DMSO", "cmpd_conc": conc, "cmpd_conc_unit": "mM", "pert_type": "pos_con"})
                    elif row in ['F', 'G', 'H']:
                        conc = concentrations[idx]
                        rec.update({"cmpdname": "fenbendazole", "solvent": "DMSO", "cmpd_conc": conc, "cmpd_conc_unit": "mM", "pert_type": "pos_con"})
                    elif row in ['I', 'J', 'K', 'L', 'M', 'N']:
                        rec.update({"cmpdname": "DMSO", "solvent": "DMSO", "cmpd_conc": 0, "cmpd_conc_unit": "%", "pert_type": "neg_con"})
                
                records.append(rec)

df = pd.DataFrame(records)
df.head(20)

df.to_csv("spher-colo6-az-version5.csv")

In [ ]:
df.cp_id.unique()

In [ ]:
df.describe